In [2]:
!pip install implicit

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 9.6 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
from scipy.sparse import coo_matrix
import implicit
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv('../BigMovieData/ml-32m/movies.csv')
# Assume ratings contains userId, movieId, rating
user_map = {u: i for i, u in enumerate(ratings['userId'].unique())}
item_map = {m: i for i, m in enumerate(ratings['movieId'].unique())}

ratings['user_idx'] = ratings['userId'].map(user_map)
ratings['item_idx'] = ratings['movieId'].map(item_map)

# Create sparse user-item matrix
user_item_matrix = coo_matrix((ratings['rating'],
                               (ratings['user_idx'], ratings['item_idx'])))



# Convert to the format expected by implicit (items x users)
item_user_matrix = user_item_matrix.T.tocsr()

# Train the model
model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.1, iterations=20)

# Optionally scale confidence (for implicit feedback)
model.fit((item_user_matrix * 40).astype('double'))

user_items_csr = user_item_matrix.tocsr()

# Pick internal user index
user_index = user_map[1]

# Pass only the user's row (1D slice)
recommended = model.recommend(user_index, user_items_csr[user_index], N=10)


# Convert back to movie IDs
reverse_item_map = {v: k for k, v in item_map.items()}

print("Max key in reverse_item_map:", max(reverse_item_map.keys()))
print("Example keys:", list(reverse_item_map.keys())[:5])
print("Problematic ID:", recommended[0][0])

# recommended_movie_ids = [reverse_item_map[i] for i in recommended[0]]


# # Optionally, map to movie titles
# recommended_movies = movies[movies['movieId'].isin(recommended_movie_ids)]
# print(recommended_movies[['movieId', 'title']])


100%|██████████| 20/20 [01:59<00:00,  5.96s/it]


(array([169557,  41449,  24104,  21708,  66469,  62548, 185049, 122171,
        32496, 168574], dtype=int32), array([2.8009534, 1.5634068, 1.4188222, 1.3952271, 1.3548373, 1.3431793,
       1.3420978, 1.3252463, 1.3228093, 1.3227613], dtype=float32))
Max key in reverse_item_map: 84431
Example keys: [0, 1, 2, 3, 4]
Problematic ID: 169557


In [3]:
import pandas as pd
from scipy.sparse import coo_matrix
import implicit
from scipy.sparse import coo_matrix
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv('../BigMovieData/ml-32m/movies.csv')
# Unique users and items
unique_user_ids = ratings['userId'].unique()
unique_movie_ids = ratings['movieId'].unique()

# Build maps
user_map = {user_id: i for i, user_id in enumerate(unique_user_ids)}
item_map = {movie_id: i for i, movie_id in enumerate(unique_movie_ids)}

# Reverse map for decoding
reverse_item_map = {i: movie_id for movie_id, i in item_map.items()}

# Build interaction matrix
rows = ratings['userId'].map(user_map)
cols = ratings['movieId'].map(item_map)
data = ratings['rating'].astype(float)

user_item_matrix = coo_matrix((data, (rows, cols)), shape=(len(user_map), len(item_map)))


/home/larry/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.1, iterations=20)
item_user_matrix = user_item_matrix.T.tocsr()
# Optionally scale confidence (for implicit feedback)
model.fit((item_user_matrix * 40).astype('double'))
user_index = user_map[1]  # userId = 1
user_index = int(user_index)
recommended = model.recommend(user_index, item_user_matrix, N=10)


# Decode movieIds
recommended_movie_ids = [reverse_item_map.get(int(i)) for i in recommended[0]]
recommended_movies = movies[movies['movieId'].isin(recommended_movie_ids)]
print(recommended_movies[['movieId', 'title']])


100%|██████████| 20/20 [01:36<00:00,  4.85s/it]


ValueError: user_items must contain 1 row for every user in userids

In [9]:
print("user_index:", user_index)
print("item_user_matrix.T shape:", item_user_matrix.T.shape)


user_index: 0
item_user_matrix.T shape: (200948, 84432)
